# UniContainer — Python quickstart

`unicontainer` is a Cython extension over the UniContainer C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install lituus-unicontainer
```

CI executes this notebook against the wheel the release actually publishes, so
the outputs below are what the code produced, not what it was expected to.

## A box is a length, a kind and a payload

MP4, MOV, HEIF, AVIF and an ALAC `.m4a` are the same structure. This library
walks that structure and hands over spans; it never decodes what is inside one.

In [1]:
def box(kind, payload=b""):
    size = len(payload) + 8
    return size.to_bytes(4, "big") + kind.encode("ascii") + payload

data = box("ftyp", b"isom") + box("moov", box("trak", box("mdia", b"the payload")))
len(data)

47

## Recognising the format, and reaching a box by name

In [2]:
import unicontainer

unicontainer.is_isobmff(data)

True

In [3]:
body, body_end = unicontainer.find_box(data, "moov/trak/mdia")
data[body:body_end]

b'the payload'

A path names the way down. A box that is absent answers `None` rather than
raising: half the boxes a format defines are optional, and that is not an error.

In [4]:
print(unicontainer.find_box(data, "moov/udta"))

None


A step that cannot name a box is refused, though. A kind is four characters,
so a shorter one cannot exist and a longer one would silently never match.

In [5]:
try:
    unicontainer.find_box(data, "moo")
except unicontainer.UniContainerError as error:
    print(error)

every step of the path must be four characters: moo


## Where to look next

See `include/UniContainer.h`, and the book for the full picture.